In [2]:
## Try some examples - lets spatialize a talker and a noise source
import torch
import torchaudio
from scipy import signal
import numpy as np
import glob
import nnresample
from scipy.io.wavfile import read, write
import h5py

## EG audio data
h5_file = '/om2/user/msaddler/projects/ibmHearingAid/assets/data/datasets/JSIN_v3.00/nStim_20000/2000ms/rms_0.1/noiseSNR_-10_10/stimSR_20000/reverb_none/noise_all/JSIN_all_v3/subsets/valid_RQTTZB4C3TJJVLJUWDV72TYMC7S4MNHH/JSIN_all__run_000_RQTTZB4C3TJJVLJUWDV72TYMC7S4MNHH.h5'

dataset = h5py.File(h5_file, 'r', swmr=True)
# set handles for easy access         
speakers = dataset['sources']['signal']['speaker_int']
signals = dataset['sources']['signal']['signal'] 
noises = dataset['sources']['noise']['signal']

## Path to HRTFs 
hrtf_dir = '/om/user/francl/Room_Simulator_20181115_Rebuild/' \
        'Expanded_HRIRdist140-5deg_elev_az_room10x10y4z_materials11wall22floor20ciel/'

hrtf_paths = glob.glob(hrtf_dir + 'Expanded_HRIRdist140-5deg_elev_az_room10x10y4z_materials11wall22floor20ciel/*')


locations = np.unique([loc[:-6] for loc in os.listdir(hrtf_dir)])


# pick eg location for signal
location = locations[10]

left = Path(hrtf_dir, location + '_l.wav').as_posix()
right = Path(hrtf_dir, location + '_r.wav').as_posix()

# pick eg location for noise 
noise_loc = locations[220]

n_left = Path(hrtf_dir, noise_loc + '_l.wav').as_posix()
n_right = Path(hrtf_dir, noise_loc + '_r.wav').as_posix()

talker = dataset['sources']['signal']['signal'][0]
distractor = dataset['sources']['noise']['signal'][0]

# normalize audio input level
level_norm = 0.02

talker = talker/talker.max() * level_norm

rate_out = 44100
rate_in = 20000
nroutsamples = round(len(talker) * rate_out/rate_in)

up_speech = signal.resample(talker, nroutsamples)

up_noise = signal.resample(distractor, nroutsamples)
# low-pass filter to remove artifacts from upsampling 

b,a = signal.butter(8, 10000, 'low',  fs=44100)
# scipy_filtered = signal.sosfilt(sos, up_speech)
scipy_filtered = signal.filtfilt(b,a, up_speech)

noise_filtered = signal.filtfilt(b,a, up_noise)


#Reads in HRIRs
hrir_r =  read(right)[1].astype(np.float32)
hrir_l = read(left)[1].astype(np.float32)

hrir_r_n =  read(n_right)[1].astype(np.float32)
hrir_l_n  = read(n_left)[1].astype(np.float32) 

#"spatializes" the sound, float64 return value. VERY loud. Do not play.
conv_stim_r = signal.fftconvolve(scipy_filtered,hrir_r)
conv_stim_l = signal.fftconvolve(scipy_filtered,hrir_l) #don't need to specifically use fft convolve 

conv_noise_r = signal.fftconvolve(noise_filtered,hrir_r_n)
conv_noise_l = signal.fftconvolve(noise_filtered,hrir_l_n)

#Rescale to not blow out headphones/eardrums
max_val = max(np.max(conv_stim_r),np.max(conv_stim_l))
rescaled_conv_stim_r = conv_stim_r/max_val*level_norm
rescaled_conv_stim_l = conv_stim_l/max_val*level_norm

max_val = max(np.max(conv_noise_r),np.max(conv_noise_l))
rescaled_conv_noise_r = conv_noise_r/max_val*level_norm
rescaled_conv_noise_l = conv_noise_l/max_val*level_norm

binaural_talker = np.array([rescaled_conv_stim_l, rescaled_conv_stim_r], dtype=np.float32).T

binaural_noise = np.array([rescaled_conv_noise_l, rescaled_conv_noise_r], dtype=np.float32).T

NameError: name 'locations' is not defined

In [ ]:
def spatialize_sound(y, brir):
    """
    This takes a left-aligned BRIR and convolves it with a left-padded signal
    (using "valid" padding) to produce the same output as "same" padding with
    a center-aligned BRIR. It is faster to pad the signal than it is to pad
    the BRIR.
    
    Args
    ----
    y (np.ndarray): monoaural waveform with shape [timesteps]
    brir (np.ndarray): binaural room impulse response with shape [timesteps, 2]
    
    Returns
    -------
    y_spatialized (np.ndarray): binaural waveform with shape [timesteps, 2]
    """
    y_padded = np.pad(y, (brir.shape[0] - 1, 0))
    y_spatialized_l = scipy.signal.convolve(y_padded, brir[:, 0], mode='valid', method='direct')
    y_spatialized_r = scipy.signal.convolve(y_padded, brir[:, 1], mode='valid', method='direct')
    return np.stack([y_spatialized_l, y_spatialized_r]).T